# Appending Zarr v2 slices with time-series-friendly chunks

This notebook generates and appends 31 daily Zarr v2 slices in `inputs/zarr-v2`. New configurations use Zarr v3 by default, so these calls explicitly set `zarr_format=2`. The output is written below `outputs`, which is ignored by Git.

In [1]:
import os
import shutil
from pathlib import Path

import xarray as xr
from create_inputs import make_zarr_v2_slices

from zappend.api import zappend

## Generate daily input slices

The helper creates deterministic daily slices with dimensions `(time=1, lat=1800, lon=3600)`. Each day has a distinct constant value, which makes it easy to check that all inputs were appended in order.

In [2]:
%%time
make_zarr_v2_slices(31)

CPU times: user 10.8 s, sys: 1.98 s, total: 12.7 s
Wall time: 16.2 s


## Collect the inputs

`zappend` accepts paths directly. Sorting the paths puts the daily slices in chronological order. The same list is used for both target layouts below.

In [3]:
input_dir = Path("inputs/zarr-v2")
output_dir =Path("outputs")
output_dir.mkdir(exist_ok=True)
slices = sorted(str(path) for path in input_dir.glob('*.zarr'))
slices

['inputs/zarr-v2/slice-2024-01-01.zarr',
 'inputs/zarr-v2/slice-2024-01-02.zarr',
 'inputs/zarr-v2/slice-2024-01-03.zarr',
 'inputs/zarr-v2/slice-2024-01-04.zarr',
 'inputs/zarr-v2/slice-2024-01-05.zarr',
 'inputs/zarr-v2/slice-2024-01-06.zarr',
 'inputs/zarr-v2/slice-2024-01-07.zarr',
 'inputs/zarr-v2/slice-2024-01-08.zarr',
 'inputs/zarr-v2/slice-2024-01-09.zarr',
 'inputs/zarr-v2/slice-2024-01-10.zarr',
 'inputs/zarr-v2/slice-2024-01-11.zarr',
 'inputs/zarr-v2/slice-2024-01-12.zarr',
 'inputs/zarr-v2/slice-2024-01-13.zarr',
 'inputs/zarr-v2/slice-2024-01-14.zarr',
 'inputs/zarr-v2/slice-2024-01-15.zarr',
 'inputs/zarr-v2/slice-2024-01-16.zarr',
 'inputs/zarr-v2/slice-2024-01-17.zarr',
 'inputs/zarr-v2/slice-2024-01-18.zarr',
 'inputs/zarr-v2/slice-2024-01-19.zarr',
 'inputs/zarr-v2/slice-2024-01-20.zarr',
 'inputs/zarr-v2/slice-2024-01-21.zarr',
 'inputs/zarr-v2/slice-2024-01-22.zarr',
 'inputs/zarr-v2/slice-2024-01-23.zarr',
 'inputs/zarr-v2/slice-2024-01-24.zarr',
 'inputs/zarr-v2

## Default layout: favor spatial access

This first call keeps the input encoding. The source data use one day per time chunk and 450 by 450 spatial chunks, a sensible layout when maps or single days are the common access pattern. `force_new=True` makes the notebook repeatable by replacing only this example target.

In [9]:
%%time
full_time_chunk = len(slices)
target = f"{output_dir}/v2_example_space_opt.zarr"
count = zappend(
    slices,
    target_dir=target,
    zarr_format=2,
    force_new=True,
    logging="INFO",
)
assert count == full_time_chunk

2026-11-02 16:21:30,553 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-11-02 16:21:30,554 WARNING Permanently deleting outputs/v2_example_space_opt.zarr
2026-11-02 16:21:30,678 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-01.zarr
2026-11-02 16:21:30,748 INFO Creating target dataset outputs/v2_example_space_opt.zarr
2026-11-02 16:21:31,038 INFO Transaction completed.
2026-11-02 16:21:31,039 INFO Slice dataset inputs/zarr-v2/slice-2024-01-01.zarr closed
2026-11-02 16:21:31,065 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-02.zarr
2026-11-02 16:21:31,100 INFO Updating target dataset outputs/v2_example_space_opt.zarr
2026-11-02 16:21:31,497 INFO Transaction completed.
2026-11-02 16:21:31,498 INFO Slice dataset inputs/zarr-v2/slice-2024-01-02.zarr closed
2026-11-02 16:21:31,523 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-03.zarr
2026-11-02 16:21:31,562 INFO Updating target dataset outp

CPU times: user 15.3 s, sys: 3.25 s, total: 18.6 s
Wall time: 14.3 s


Opening the target shows its dimensions and the chunks inherited from the first slice. Compare this with the second target below.

In [10]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 51.3 ms, sys: 18.4 ms, total: 69.7 ms
Wall time: 54.5 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

## Time-series layout

Here the data variables are written with one 31-day time chunk. A request for a point time series therefore reads one temporal chunk instead of 31 daily chunks. This trades more data read for map requests against fewer reads for time-series requests.

In [11]:
%%time
full_time_chunk = len(slices)
target = f"{output_dir}/v2_example_time_opt.zarr"
count = zappend(
    slices,
    target_dir=target,
    zarr_format=2,
    force_new=True,
    variables={
        'temperature': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
        'quality_flag': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
    },
    logging="INFO",
)
assert count == full_time_chunk

2026-11-02 16:21:45,049 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-11-02 16:21:45,050 WARNING Permanently deleting outputs/v2_example_time_opt.zarr
2026-11-02 16:21:45,116 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-01.zarr
2026-11-02 16:21:45,149 INFO Creating target dataset outputs/v2_example_time_opt.zarr
2026-11-02 16:21:49,874 INFO Transaction completed.
2026-11-02 16:21:49,875 INFO Slice dataset inputs/zarr-v2/slice-2024-01-01.zarr closed
2026-11-02 16:21:49,902 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-02.zarr
2026-11-02 16:21:49,945 INFO Updating target dataset outputs/v2_example_time_opt.zarr
2026-11-02 16:21:56,212 INFO Transaction completed.
2026-11-02 16:21:56,213 INFO Slice dataset inputs/zarr-v2/slice-2024-01-02.zarr closed
2026-11-02 16:21:56,241 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-03.zarr
2026-11-02 16:21:56,295 INFO Updating target dataset outputs

CPU times: user 4min 35s, sys: 1min 15s, total: 5min 50s
Wall time: 3min 53s


In [12]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 51.4 ms, sys: 10.2 ms, total: 61.6 ms
Wall time: 52.1 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

In [13]:
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)